# Étape 7 – Figures

Notebook de lecture de `src/figures.py` : même code, découpé en cellules. Exécuter les cellules dans l'ordre équivaut à `python -m src.figures`.

Figures dédiées à la présentation.

Lit uniquement des sorties existantes (aucun recalcul) :
    outputs/access_summary.json            -> part sans option par décile de revenu, part desservie par mode
    outputs/recommendations_summary.json   -> stratégie recommandée vs alternatives (même budget Citiz)
    outputs/territories_citiz.csv          -> classement national des territoires pour Citiz
    outputs/frequency_<zone>.json          -> comptage d'arrêts vs fréquence des lignes (slides_frequency.png)
    outputs/usage_validation.json          -> occupation réelle par quintile de propension (slides_usage.png)
    outputs/equity_curve.json              -> prix de l'équité, territoire prioritaire (slides_equity.png)

Sorties : outputs/figures/slides_deciles.png, slides_modes.png, slides_strategies.png,
          slides_territories.png, slides_map_france.png, slides_map_top.png (territoire prioritaire),
          slides_map_ems.png (témoin) ; lisent aussi cells_access.parquet, recommendations*.csv
Usage : python -m src.figures

In [ ]:
# Fabric : à exécuter en premier. En local (hors Fabric), cette cellule ne fait rien.
import os
import subprocess
import sys

FABRIC_PROJECT = "/lakehouse/default/Files/mobiliscore"  # dossier du projet uploadé dans le Lakehouse
if os.path.isdir("/lakehouse/default"):
    os.chdir(FABRIC_PROJECT)  # la préparation trouve Config.py dans le dossier courant
    # inutile si un Environment Fabric fournit déjà ces paquets (recommandé : voir FABRIC.md)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj>=3.6", "openpyxl>=3.1"], check=True)

In [ ]:
# Préparation : ce notebook exécute le code de src/figures.py tel quel, depuis la racine du projet.
import ast
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "Config.py").exists():
    ROOT = ROOT.parent
__file__ = str(ROOT / "src" / "figures.py")
__doc__ = ast.get_docstring(ast.parse(Path(__file__).read_text(encoding="utf-8")))
sys.argv = ["src.figures"]  # options par défaut, comme `python -m src.figures`

## Imports et constantes

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import Config as C  # noqa: E402

BG = "#fbfaf8"
INK = "#1f2328"
MUTED = "#6b7280"
GRID = "#e5e7eb"
BLUE = "#2b76d6"
ORANGE = "#e8743b"
GREY = "#b8c2cc"

### Fonction `_style`

In [ ]:
def _style(ax):
    ax.set_facecolor(BG)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

### Fonction `_fig`

In [ ]:
def _fig(w=10, h=5):
    fig, ax = plt.subplots(figsize=(w, h), dpi=150)
    fig.patch.set_facecolor(BG)
    _style(ax)
    return fig, ax

### Fonction `deciles`

In [ ]:
def deciles(acc):
    d = acc["by_income_decile"]
    keys = sorted(d, key=int)
    vals = [100 * d[k]["share_no_option"] for k in keys]
    nat = 100 * acc["national"]["share_no_option"]
    fig, ax = _fig()
    colors = [ORANGE if v == max(vals) else BLUE for v in vals]
    bars = ax.bar(range(len(keys)), vals, color=colors, width=0.62)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.0f} %", ha="center", color=INK, fontsize=10)
    ax.axhline(nat, color=MUTED, linestyle=":", linewidth=1)
    ax.text(len(keys) - 0.5, nat + 0.8, f"France : {nat:.0f} %", ha="right", color=MUTED, fontsize=9)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels([f"D{k}" for k in keys], color=INK)
    ax.set_xlabel("décile de revenu par habitant du carreau (D1 = 10 % les plus modestes)", color=MUTED)
    ax.set_ylabel("% d'habitants sans aucune option à 5 min", color=MUTED)
    ax.set_ylim(0, max(vals) * 1.18)
    ax.set_title("Qui est concerné ? Surtout les déciles de revenu intermédiaires (courbe en U)",
                 loc="left", color=INK, fontsize=12)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_deciles.png", facecolor=BG)
    plt.close(fig)

### Fonction `modes`

In [ ]:
def modes(acc):
    sh = acc["national"]["share_served_by_mode"]
    order = sorted(sh, key=lambda m: sh[m])
    vals = [100 * sh[m] for m in order]
    fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
    fig.patch.set_facecolor(BG)
    _style(ax)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    colors = [ORANGE if m == "citiz" else BLUE for m in order]
    bars = ax.barh(range(len(order)), vals, color=colors, height=0.6)
    for b, v in zip(bars, vals):
        ax.text(v + 0.8, b.get_y() + b.get_height() / 2, f"{v:.1f} %".replace(".", ","), va="center", color=INK, fontsize=10)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([C.MODES[m] for m in order], color=INK)
    ax.set_xlabel("% de la population (France + Martinique + La Réunion) ayant ≥ 1 point du mode à 5 min",
                  color=MUTED)
    ax.set_xlim(0, max(vals) * 1.15)
    ax.set_title("Accès à 5 min de marche, par mode", loc="left", color=INK, fontsize=12)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_modes.png", facecolor=BG)
    plt.close(fig)

### Fonction `_num`

Nombre au format français (virgule décimale, zéros inutiles retirés).

In [ ]:
def _num(v, dec=0):
    """Nombre au format français (virgule décimale, zéros inutiles retirés)."""
    t = f"{float(v):.{dec}f}"
    if "." in t:
        t = t.rstrip("0").rstrip(".")
    return t.replace(".", ",")

### Fonction `_strategy`

Bloc de comparaison sous l'un des noms possibles (schéma évolutif), sinon None.

In [ ]:
def _strategy(rec, exact, words):
    """Bloc de comparaison sous l'un des noms possibles (schéma évolutif), sinon None."""
    comp = rec.get("comparison") or {}
    for k in exact:
        if isinstance(comp.get(k), dict):
            return comp[k]
    for k, v in comp.items():
        if isinstance(v, dict) and all(w in k.lower() for w in words):
            return v
    return None

### Fonction `strategies`

Stratégies comparées, volet Citiz d'un territoire (même budget Citiz).

In [ ]:
def strategies(full):
    """Stratégies comparées, volet Citiz d'un territoire (même budget Citiz)."""
    rec = (full.get("funders") or {}).get("citiz") or full
    sel = rec.get("selected") or {}
    lam = rec.get("equity_lambda", C.EQUITY_LAMBDA)
    rows = [
        (f"Recommandée\n(modèle, λ = {_num(lam, 1)},\nquota peu dense)", sel, True),
        ("Offre existante,\nsans modèle (v = 1)",
         _strategy(rec, ["existing_offer_no_model", "no_model_value_1", "no_model"], ["no_model"]), False),
        ("Sans pondération\nd'équité (λ = 0)",
         _strategy(rec, ["no_equity_weight", "lambda_0_no_equity", "no_equity", "lambda_0"], ["equity"]), False),
        ("Naïve\n(population seule)", _strategy(rec, ["naive_population_only"], ["naive"]), False),
    ]
    rows = [r for r in rows if r[1]]
    metrics = [
        ("pop_newly_served", "Habitants nouvellement\ndesservis", lambda v: f"{v:,.0f}".replace(",", " ")),
        ("co2_avoided_t_yr", "CO2 évité (t/an)", lambda v: f"{v:.0f}"),
        ("share_new_in_income_deciles_1_3", "Part des nouveaux desservis\ndans les déciles 1-3 (%)",
         lambda v: f"{v:.0f}"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(4.4 * max(len(rows), 3), 4.8), dpi=150)
    fig.patch.set_facecolor(BG)
    for ax, (key, title, fmt) in zip(axes, metrics):
        _style(ax)
        vals = [(r[1].get(key) or 0) * (100 if key.startswith("share") else 1) for r in rows]
        colors = [BLUE if r[2] else GREY for r in rows]
        bars = ax.bar(range(len(rows)), vals, color=colors, width=0.6)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v * 1.02, fmt(v), ha="center", va="bottom",
                    color=INK, fontsize=10)
        ax.set_xticks(range(len(rows)))
        ax.set_xticklabels([r[0] for r in rows], fontsize=8, color=INK)
        ax.set_ylim(0, max(max(vals), 1e-9) * 1.2)
        ax.set_yticks([])
        ax.set_title(title, color=INK, fontsize=11)
    budget = rec.get("budget_eur", C.DEFAULT_BUDGET) / 1e6
    fig.suptitle(f"{full.get('territoire', '')} – stations Citiz, même budget "
                 f"({_num(budget, 2)} M€ sur {C.HORIZON_YEARS} ans)",
                 x=0.01, ha="left", color=INK, fontsize=12)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_strategies.png", facecolor=BG)
    plt.close(fig)

In [ ]:
# Rampe froid -> chaud identique à l'application (src/app_data.RAMP) ; 0 option = gris neutre.
RAMP = [(49, 54, 149), (69, 117, 180), (116, 173, 209), (171, 217, 233), (254, 224, 144),
        (253, 174, 97), (244, 109, 67), (215, 48, 39), (165, 0, 38)]
ZERO = (120 / 255, 130 / 255, 150 / 255)

### Fonction `_cmap`

In [ ]:
def _cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("froid_chaud", [tuple(c / 255 for c in r) for r in RAMP])

### Fonction `map_france`

Carte métropole : score moyen pondéré par la population sur une grille de 2 km.

In [ ]:
def map_france():
    """Carte métropole : score moyen pondéré par la population sur une grille de 2 km."""
    import pandas as pd
    ca = pd.read_parquet(C.OUT / "cells_access.parquet", columns=["crs_id", "x", "y", "ind", "score"])
    ca = ca[ca.crs_id == 0]
    g = 2000
    ca = ca.assign(gx=(ca.x // g).astype(int), gy=(ca.y // g).astype(int), ws=ca.score * ca.ind,
                   none=(ca.score <= 0) * ca.ind)
    a = ca.groupby(["gx", "gy"], as_index=False)[["ind", "ws", "none"]].sum()
    a["s"] = a.ws / a.ind
    a["zero"] = a.none / a.ind > 0.5           # majorité des habitants sans option
    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    z = a[a.zero]
    ax.scatter(z.gx * g, z.gy * g, s=1.3, marker="s", color=ZERO, linewidths=0)
    nz = a[~a.zero].sort_values("s")
    sc = ax.scatter(nz.gx * g, nz.gy * g, c=nz.s, cmap=_cmap(), vmin=0, vmax=100, s=1.3, marker="s",
                    linewidths=0)
    ax.set_aspect("equal")
    ax.axis("off")
    cb = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)
    cb.set_label("score d'accès 0-100 (froid -> chaud)", color=MUTED)
    cb.outline.set_visible(False)
    ax.set_title("Accès à la mobilité à 5 min de marche (carrés de 2 km ; gris = majorité sans option)",
                 loc="left", color=INK, fontsize=10)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_map_france.png", facecolor=BG)
    plt.close(fig)

### Fonction `territories`

Classement national : adhérents potentiels (modèle) et retenus après risque terrain (+ Strasbourg témoin).

In [ ]:
def territories():
    """Classement national : adhérents potentiels (modèle) et retenus après risque terrain (+ Strasbourg témoin)."""
    import pandas as pd
    t = pd.read_csv(C.OUT / "territories_citiz.csv")
    t = t[t["rank"].notna() | t["reference"]].copy()
    t = t.sort_values("sim_members_adjusted", ascending=True)
    col = {"nouveau territoire": ORANGE, "extension": BLUE}
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.4), dpi=150)
    fig.patch.set_facecolor(BG)
    _style(ax)
    colors = [GREY if r else col.get(s_, BLUE) for r, s_ in zip(t["reference"], t["statut"])]
    labels = [f"{n}{' (témoin)' if r else ''}" for n, r in zip(t["territoire"], t["reference"])]
    ax.barh(labels, t["sim_members_potential"], color=colors, alpha=0.25, height=0.6)
    bars = ax.barh(labels, t["sim_members_adjusted"], color=colors, height=0.6)
    for b, v, raw, rk in zip(bars, t["sim_members_adjusted"], t["sim_members_potential"], t["risk"]):
        note = f"  (modèle {raw:,.0f} · risque {100 * rk:+.0f} %)" if rk else ""
        ax.text(max(v, raw) * 1.01, b.get_y() + b.get_height() / 2,
                f"{v:,.0f} adhérents{note}".replace(",", " "), va="center", color=INK, fontsize=8)
    ax.set_xlim(0, t["sim_members_potential"].max() * 1.75)
    ax.set_xticks([])
    ax.tick_params(axis="y", labelsize=9, colors=INK)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=ORANGE, label="Citiz absent (nouveau territoire)"),
                       Patch(color=BLUE, label="Citiz présent (extension)"),
                       Patch(color=GREY, label="Strasbourg, territoire témoin"),
                       Patch(color=GREY, alpha=0.25, label="potentiel du modèle avant risque terrain")],
              loc="lower right", frameon=False, fontsize=8)
    ax.set_title(f"Même budget Citiz ({_num(C.CITIZ_BUDGET / 1e6, 2)} M€ / {C.HORIZON_YEARS} ans) : "
                 "adhérents retenus après risque terrain", loc="left", color=INK, fontsize=11)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_territories.png", facecolor=BG)
    plt.close(fig)

### Fonction `map_area`

Territoire : score au carreau 200 m + sites recommandés par type et par financeur.

In [ ]:
def map_area(csv, summary, out_name, communes):
    """Territoire : score au carreau 200 m + sites recommandés par type et par financeur."""
    import pandas as pd
    ca = pd.read_parquet(C.OUT / "cells_access.parquet", columns=["lon", "lat", "score", "com"],
                         filters=[("com", "in", list(communes))])
    rec = pd.read_csv(C.OUT / csv)
    # cadrage sur les sites recommandés (+ ~5 km) : l'arrière-pays peu peuplé écraserait la carte
    w, e = rec.lon.min() - 0.06, rec.lon.max() + 0.06
    s_, n = rec.lat.min() - 0.045, rec.lat.max() + 0.045
    ca = ca[ca.lon.between(w, e) & ca.lat.between(s_, n)]
    fig, ax = plt.subplots(figsize=(8, 7.2), dpi=150)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    z = ca[ca.score <= 0]
    ax.scatter(z.lon, z.lat, s=8, marker="s", color=ZERO, linewidths=0, alpha=0.8)
    nz = ca[ca.score > 0]
    ax.scatter(nz.lon, nz.lat, c=nz.score, cmap=_cmap(), vmin=0, vmax=100, s=8, marker="s",
               linewidths=0, alpha=0.8)
    styles = {"citiz": ("o", "#e8743b", "Station Citiz (Citiz)"),
              "bike": ("^", "#1a9e5c", "Station vélo (collectivité)"),
              "bus": ("s", "#5b3fb5", "Arrêt de bus (collectivité)")}
    for k, (m, col, lab) in styles.items():
        r = rec[rec.type == k]
        ax.scatter(r.lon, r.lat, marker=m, s=90, color=col, edgecolors="white", linewidths=1.5,
                   label=f"{lab} : {len(r)}", zorder=3)
    ax.set_aspect(1 / np.cos(np.radians(ca.lat.median())))
    ax.axis("off")
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor=GRID, fontsize=9)
    fu = summary.get("funders") or {}
    bc = (fu.get("citiz") or {}).get("budget_eur", C.CITIZ_BUDGET)
    bp = (fu.get("collectivite") or {}).get("budget_eur", C.PUBLIC_BUDGET)
    ax.set_title(f"{summary.get('territoire', '')} : budget Citiz {_num(bc / 1e6, 2)} M€ · "
                 f"collectivité {_num(bp / 1e6, 2)} M€", loc="left", color=INK, fontsize=11)
    fig.tight_layout()
    fig.savefig(C.FIG / out_name, facecolor=BG)
    plt.close(fig)

In [ ]:
ZONE_LABEL = {"ems": "Eurométropole de Strasbourg", "epci200030195": "Métropole Nice Côte d'Azur",
              "epci200046977": "Métropole de Lyon"}

### Fonction `frequency`

Comptage d'arrêts vs fréquence des lignes dans les zones d'étude (outputs/frequency_<zone>.json).

In [ ]:
def frequency():
    """Comptage d'arrêts vs fréquence des lignes dans les zones d'étude (outputs/frequency_<zone>.json)."""
    zones = [(n, json.loads((C.OUT / f"frequency_{n}.json").read_text(encoding="utf-8")))
             for n in C.FREQ_FEEDS if (C.OUT / f"frequency_{n}.json").exists()]
    zones = [(n, s) for n, s in zones if s.get("comparaison")]
    if not zones:
        return None
    bars = [("part_pop_arret_tc_5min", "Un arrêt à 5 min\n(comptage, méthode nationale)", GREY),
            ("part_pop_au_moins_1_option_tc_pleine", "Au moins une ligne toutes les 15 min\n(fréquence GTFS)", BLUE),
            ("part_pop_arret_mais_service_faible", "Un arrêt, mais service faible", ORANGE)]
    fig, ax = _fig(10, 4.6)
    w = 0.26
    for k, (key, lab, col) in enumerate(bars):
        vals = [100 * s["comparaison"][key] for _, s in zones]
        xs = np.arange(len(zones)) + (k - 1) * w
        b = ax.bar(xs, vals, width=w * 0.92, color=col, label=lab.replace("\n", " "))
        for r, v in zip(b, vals):
            ax.text(r.get_x() + r.get_width() / 2, v + 1, f"{_num(v, 1)} %", ha="center", color=INK, fontsize=9)
    ax.set_xticks(np.arange(len(zones)))
    ax.set_xticklabels([ZONE_LABEL.get(n, n) for n, _ in zones], color=INK)
    ax.set_ylim(0, 110)
    ax.set_ylabel("% des habitants (bus, tram, métro à 5 min)", color=MUTED)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False, fontsize=8.5)
    ax.set_title("Compter les arrêts surestime l'offre : un arrêt à 2 bus par jour y vaut une station de tram",
                 loc="left", color=INK, fontsize=12)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_frequency.png", facecolor=BG)
    plt.close(fig)
    return "slides_frequency"

### Fonction `usage`

Occupation réelle des stations Citiz par quintile de propension du modèle (outputs/usage_validation.json).

In [ ]:
def usage():
    """Occupation réelle des stations Citiz par quintile de propension du modèle (outputs/usage_validation.json)."""
    f = C.OUT / "usage_validation.json"
    if not f.exists():
        return None
    u = json.loads(f.read_text(encoding="utf-8"))
    q = u["occupation_par_quintile_de_propension"]
    vals = [100 * r["occupation_moyenne"] for r in q]
    fig, ax = _fig(10, 4.6)
    colors = [ORANGE if v == max(vals) else BLUE for v in vals]
    b = ax.bar(range(len(q)), vals, color=colors, width=0.6)
    for r, v, row in zip(b, vals, q):
        ax.text(r.get_x() + r.get_width() / 2, v + 0.8, f"{_num(v, 1)} %", ha="center", color=INK, fontsize=10)
    ax.set_xticks(range(len(q)))
    ax.set_xticklabels([f"{r['quintile']}\n({r['stations']} stations)" for r in q], color=INK, fontsize=9)
    ax.set_xlabel("quintile de propension du modèle Citiz, hors pli (Q1 = sites jugés les moins favorables)",
                  color=MUTED)
    ax.set_ylabel("occupation moyenne (% du temps-voiture réservé)", color=MUTED)
    ax.set_ylim(0, max(vals) * 1.25)
    per = u.get("periode", {})
    ax.set_title(f"Usage réel de {u['stations_validees']:,} stations Citiz (collecte GBFS, {per.get('releves')} relevés) : "
                 f"Spearman modèle {_num(u['spearman_modele_occupation'], 2)} · population seule "
                 f"{_num(u['spearman_population_occupation'], 2)}".replace(",", " ", 1),
                 loc="left", color=INK, fontsize=10.5)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_usage.png", facecolor=BG)
    plt.close(fig)
    return "slides_usage"

### Fonction `equity`

Le prix de l'équité sur le territoire prioritaire (outputs/equity_curve.json, src/recommend.py).

In [ ]:
def equity():
    """Le prix de l'équité sur le territoire prioritaire (outputs/equity_curve.json, src/recommend.py)."""
    f = C.OUT / "equity_curve.json"
    if not f.exists():
        return None
    e = json.loads(f.read_text(encoding="utf-8"))
    from matplotlib.ticker import FuncFormatter
    panels = [(k, col, lab) for k, col, lab in (("citiz", ORANGE, "Citiz (stations d'autopartage)"),
                                                 ("collectivite", BLUE, "Collectivité (vélo, bus)"))
              if any(p["financeur"] == k for p in e["points"])]
    fig, axes = plt.subplots(1, len(panels), figsize=(11, 4.6), dpi=150, squeeze=False)
    fig.patch.set_facecolor(BG)
    for ax, (key, col, lab) in zip(axes[0], panels):
        _style(ax)
        ax.grid(axis="x", color=GRID, linewidth=0.8)
        pts = sorted((p for p in e["points"] if p["financeur"] == key), key=lambda p: p["lambda"])
        x = [100 * p["share_new_in_income_deciles_1_3"] for p in pts]
        y = [p["pop_newly_served"] for p in pts]
        ax.plot(x, y, color=col, linewidth=2, marker="o")
        # une étiquette par point distinct : « λ = 0,5 · 0,75 · 1 » si plusieurs λ donnent la même liste
        groups = {}
        for p, xi, yi in zip(pts, x, y):
            groups.setdefault((round(xi, 2), round(yi)), []).append(_num(p["lambda"], 2))
        for (xi, yi), lams in groups.items():
            ax.annotate("λ = " + " · ".join(lams), (xi, yi), textcoords="offset points", xytext=(6, 8),
                        color=col, fontsize=9)
        pad_x = max(1.0, (max(x) - min(x)) * 0.25)
        pad_y = max(500, (max(y) - min(y)) * 0.6)
        ax.set_xlim(min(x) - pad_x, max(x) + pad_x * 1.6)
        ax.set_ylim(min(y) - pad_y, max(y) + pad_y)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " ")))
        ax.set_xlabel("% des nouveaux desservis dans les déciles de revenu 1-3", color=MUTED)
        ax.set_ylabel("habitants nouvellement desservis", color=MUTED)
        ax.set_title(lab, loc="left", color=col, fontsize=11)
    fig.suptitle(f"{e.get('territoire', '')} : le prix de l'équité (même budget, λ de 0 à 1)",
                 x=0.01, ha="left", color=INK, fontsize=12)
    fig.tight_layout()
    fig.savefig(C.FIG / "slides_equity.png", facecolor=BG)
    plt.close(fig)
    return "slides_equity"

### Fonction `main`

In [ ]:
def main():
    acc = json.loads((C.OUT / "access_summary.json").read_text(encoding="utf-8"))
    rec = json.loads((C.OUT / "recommendations_summary.json").read_text(encoding="utf-8"))
    deciles(acc)
    modes(acc)
    ems_f = C.OUT / "recommendations_summary_ems.json"
    # comparaison des stratégies sur le témoin : là où l'offre existe, l'ignorer coûte cher
    strategies(json.loads(ems_f.read_text(encoding="utf-8")) if ems_f.exists() else rec)
    territories()
    map_france()
    from src import territories as T
    top = T.load_summary()["top"]["epci"]
    map_area("recommendations.csv", rec, "slides_map_top.png", T.epci_communes(top))
    if ems_f.exists():
        map_area("recommendations_ems.csv", json.loads(ems_f.read_text(encoding="utf-8")), "slides_map_ems.png", C.EMS_COMMUNES)
    extra = [n for n in (frequency(), usage(), equity()) if n]
    for n in ("slides_deciles", "slides_modes", "slides_strategies", "slides_territories", "slides_map_france",
              "slides_map_top", "slides_map_ems", *extra):
        print("->", C.FIG / f"{n}.png")

## Exécution

Bloc `__main__` du module, inchangé.

In [ ]:
if __name__ == "__main__":
    main()